# Capture Workspace App Configuration

Reads the current workspace app settings (reports, users, groups) and outputs
the `workspace_app` JSON block ready to paste into `config/dev.json`, `config/uat.json`, or `config/prod.json`.

**Steps:**
1. Set your workspace ID below
2. Run all cells
3. Copy the JSON output from the last cell into your config file

## 1. Configuration

In [ ]:
# === CONFIGURATION ===
WORKSPACE_ID = "68c48a5c-8b88-4559-962d-5dc7f29ab96a"  # <-- Set your workspace ID

PBI_API = "https://api.powerbi.com/v1.0/myorg"

print(f"Target workspace: {WORKSPACE_ID}")

## 2. Authentication

In [ ]:
import requests
import json

try:
    from notebookutils import mssparkutils
    access_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    print("\u2713 Authenticated via Fabric notebook token")
except ImportError:
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    access_token = token.token
    print("\u2713 Authenticated via DefaultAzureCredential")

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

def api_get(url):
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.ok:
        return resp.json()
    print(f"  \u2717 GET {resp.status_code}: {resp.text[:300]}")
    return None

ws_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}")
if ws_info:
    print(f"\u2713 Connected to workspace: {ws_info.get('name', 'unknown')}")

## 3. List workspace reports

In [ ]:
# Get all reports in the workspace
reports_data = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports")
ws_reports = reports_data.get("value", []) if reports_data else []
report_name_to_id = {r["name"]: r["id"] for r in ws_reports}

print(f"Found {len(ws_reports)} reports in workspace:\n")
for r in sorted(ws_reports, key=lambda x: x["name"]):
    rtype = r.get("reportType", "PowerBI")
    print(f"  {r['name']}  ({rtype})  id={r['id']}")

## 4. Find the workspace app and its config

In [ ]:
# Find the app published from this workspace
apps_data = api_get(f"{PBI_API}/apps")
all_apps = apps_data.get("value", []) if apps_data else []

workspace_app = None
for app in all_apps:
    if app.get("workspaceId") == WORKSPACE_ID:
        workspace_app = app
        break

if workspace_app:
    app_id = workspace_app["id"]
    print(f"\u2713 Found app: {workspace_app.get('name', 'unknown')}")
    print(f"  App ID: {app_id}")
    print(f"  Description: {workspace_app.get('description', '(none)')}")
else:
    app_id = None
    print("\u2139\ufe0f No published app found for this workspace.")
    print("   The config output will include all workspace reports as a starting template.")

In [ ]:
# Get reports included in the app
app_report_names = set()
if app_id:
    app_reports_data = api_get(f"{PBI_API}/apps/{app_id}/reports")
    app_reports = app_reports_data.get("value", []) if app_reports_data else []
    app_report_names = {r["name"] for r in app_reports}
    print(f"Reports currently in the app ({len(app_report_names)}):\n")
    for name in sorted(app_report_names):
        print(f"  \u2713 {name}")

    # Show reports NOT in the app
    excluded = sorted(set(report_name_to_id.keys()) - app_report_names)
    if excluded:
        print(f"\nReports NOT in the app ({len(excluded)}):\n")
        for name in excluded:
            print(f"  \u2717 {name}")
else:
    print("No app — skipping app report check.")

## 5. Get app users (requires Admin API or workspace user list)

In [ ]:
# Try Admin API first (needs Power BI Service Admin or Fabric Admin role)
# Falls back to workspace users if admin API is not available

app_users = []
app_groups = []
user_source = None

if app_id:
    admin_users_data = api_get(f"{PBI_API}/admin/apps/{app_id}/users")
    if admin_users_data and "value" in admin_users_data:
        user_source = "admin API (app users)"
        for u in admin_users_data["value"]:
            ptype = u.get("principalType", "")
            identifier = u.get("emailAddress") or u.get("identifier", "")
            if ptype == "Group":
                app_groups.append({"identifier": identifier, "displayName": u.get("displayName", "")})
            elif ptype == "User":
                app_users.append({"email": identifier, "displayName": u.get("displayName", "")})
            else:
                app_users.append({"email": identifier, "displayName": u.get("displayName", ""), "type": ptype})

# Fallback: workspace users
if not user_source:
    ws_users_data = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/users")
    if ws_users_data and "value" in ws_users_data:
        user_source = "workspace users (app users not available — admin API may require elevated permissions)"
        for u in ws_users_data["value"]:
            ptype = u.get("principalType", "")
            identifier = u.get("emailAddress") or u.get("identifier", "")
            if ptype == "Group":
                app_groups.append({"identifier": identifier, "displayName": u.get("displayName", "")})
            elif ptype == "User":
                app_users.append({"email": identifier, "displayName": u.get("displayName", "")})

print(f"Source: {user_source}\n")
print(f"Users ({len(app_users)}):")
for u in app_users:
    print(f"  {u.get('displayName', '')}  <{u['email']}>")

print(f"\nGroups ({len(app_groups)}):")
for g in app_groups:
    print(f"  {g.get('displayName', '')}  id={g['identifier']}")

## 6. Generate config JSON

Outputs the `workspace_app` block ready to paste into your config file.

- If an app exists: uses the actual app reports and users/groups
- If no app: generates a template with all workspace reports in a single audience

In [ ]:
# Build the workspace_app config block
if app_report_names:
    report_list = sorted(app_report_names)
else:
    report_list = sorted(report_name_to_id.keys())

user_emails = [u["email"] for u in app_users]
group_ids = [g["identifier"] for g in app_groups]

config_block = {
    "workspace_app": {
        "enabled": True,
        "audiences": [
            {
                "name": "Default Audience",
                "reports": report_list,
                "users": user_emails,
                "groups": group_ids
            }
        ]
    }
}

output = json.dumps(config_block, indent=2)

print("=" * 60)
print("COPY THIS INTO YOUR CONFIG FILE")
print("(replace the existing \"workspace_app\" section)")
print("=" * 60)
print()
print(output)
print()
print("=" * 60)

if not app_report_names:
    print("\n\u26a0\ufe0f  No published app found — template includes ALL workspace reports.")
    print("   Edit the \"reports\" list to include only the ones you want.")

if user_source and "workspace users" in user_source:
    print("\n\u26a0\ufe0f  Users/groups came from workspace membership (not app audience).")
    print("   Review and adjust — some workspace members may not need app access.")

print("\n\u2139\ufe0f  To split into multiple audiences, duplicate the audience block")
print("   and assign different reports/users/groups to each.")